In [61]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score

In [ ]:
train_path = '/Users/filippomontecchi/Desktop/Data Science & Machine Learning Lab/Lab/Code/CORE_CODING/classification/toy_dataset/classification_dataset.csv'
train_df = pd.read_csv(train_path)
data = train_df.loc[:, 'num_1':'cat_2']
target = train_df['target']

train_df

In [ ]:
train_df.describe()

- 1000 rows and 7 col
- different types of data, which suggest:
    - numerical data: scaler
    - ordinal data: ordinal encoder
    - categorical data: OHE

- split into train and validation set
- inspect nan distribution
- search for weird nan-like values
- check distributions

- split data

In [7]:
x_train, x_val, y_train, y_val = train_test_split(data, target, test_size=0.2, shuffle=True, random_state=42)

- inspect nan

In [ ]:
x_train.isna().mean()*100

- very chill nan % --> impute 

In [ ]:
mask = x_train.isna().mean(axis=1).sort_values(ascending = False)*100 >= 50
x_train[mask]

- 121 rows has 50% missing values ... what do we do?
    - drop
    - **impute**

In [53]:
x_train.shape

(800, 6)

- look for weird nan like values
    - build ordinal values to pass to the dumb ass ordinal encoder
    - I have 2 ordinal columns, so it expects a list of lists with 2 lists inside the outer one --> ordinal values for ord col 1 and ordinal values for ord col 2

In [54]:
num_col = x_train.loc[:, 'num_1':'num_2']
ord_col = x_train.loc[:, 'ord_1':'ord_2']
cat_col = x_train.loc[:, 'cat_1':'cat_2']


ordinal_values = []
for o_c in ord_col:
    not_nan_ordinal_values = ord_col[o_c][ord_col[o_c].notna()]
    
    # print(not_nan_ordinal_values.value_counts())
    
    unique_not_nan_ordinal_values = not_nan_ordinal_values.unique()
    # print(o_c, unique_not_nan_ordinal_values)

categories = [
    ['High', 'Medium', 'Low'],  # ord_1 categories
    ['Poor', 'Average', 'Good'] # ord_2 categories
]


In [ ]:
# in general to sort lists based on a specified order

# Lista di categorie originali
original_list = ['Low', 'High', 'Medium']

# Definisci l'ordine che desideri
desired_order = ['High', 'Medium', 'Low']

# Crea un dizionario che mappa ciascuna categoria all'indice dell'ordine desiderato
order_dict = {}
for idx, value in enumerate(desired_order):
    order_dict[value] = idx

# Ora ordina la lista originale usando il dizionario
sorted_list = sorted(original_list, key=lambda x: order_dict[x])

- no weird values

In [ ]:
l = []
for c_c in cat_col:
    not_na_categories = cat_col[c_c][cat_col[c_c].notna()]
    l.append(not_na_categories.unique().tolist())

print(l)

- nothing weird, no nan-like values

- plot histograms

In [ ]:
plt.figure(figsize=(8,8))
for plt_idx, col in enumerate(num_col):
    plt.subplot(2,1,plt_idx+1)
    sns.histplot(x=num_col[col], bins='auto', kde=True)
    plt.title(f'frequency {col}')

plt.tight_layout()
plt.show()
    

# out of curiosity
plt.figure(figsize=(8,6))
sns.scatterplot(x=num_col['num_1'], y=num_col['num_2'], hue=target[num_col.index])
plt.show()


In [ ]:
plt.figure(figsize=(6,6))
for plt_idx, col in enumerate(ord_col):
    plt.subplot(2,1,plt_idx+1)
    sns.histplot(x=ord_col[col], bins='auto', kde=False)
    plt.title(f'frequency {col}')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6,6))
for plt_idx, col in enumerate(cat_col):
    plt.subplot(2,1,plt_idx+1)
    sns.histplot(x=cat_col[col], bins='auto', kde=False)
    plt.title(f'frequency {col}')

plt.tight_layout()
plt.show()

- all the probability distributions are regular, no anomalies

- manage Nas based on workflow:
    - numerical data: scaler --> simple imputer median + add flag col
    - ordinal data: ordinal encoder --> (compute ordinal values to pass to the dumb ordinal encoder) simple imputer constant: nan-->'unknown'  + add flag col --> the encoder will map 'unknown' to -1 and the eventual missing categories to -2
    - categorical data: OHE --> simple imputer constant: nan-->'unknown' + add flag col

- tie with columntransformer REMEMBER THE PASSTHROUGH
- run some standard classification models, choose the one that get highest score withou tuning

In [49]:
num_col_name = num_col.columns.tolist()
ord_col_name = ord_col.columns.tolist()
cat_col_name = cat_col.columns.tolist()

In [64]:
num_pipe = Pipeline(steps=[
    (
        'num_imputer',
        SimpleImputer(strategy='median', add_indicator=True)
    )
    ,
    (
        'scaler',
        StandardScaler()
    )
])

ord_pipe = Pipeline(steps=[
    (
        'ord_imputer',
        SimpleImputer(strategy='constant', fill_value='unknown')
    )
    ,
    (
        'OE',
        OrdinalEncoder(categories=categories, handle_unknown='use_encoded_value', unknown_value=-1, encoded_missing_value=-2)
    )
])

cat_pipe = Pipeline(steps=[
    (
        'cat_imputer',
        SimpleImputer(strategy='constant', fill_value='unknown', add_indicator=True)
    )
    ,
    (
        'OHE',
        OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    )
])

preprocesing_pipeline = ColumnTransformer(transformers=[
    ('num_prep', num_pipe, num_col_name),
    ('ord_prep', ord_pipe, ord_col_name),
    ('cat_prep', cat_pipe, cat_col_name)
], remainder='passthrough', n_jobs=-1)

full_pipeline = Pipeline(steps=[
    ('preprocessing', preprocesing_pipeline),
    ('classifier', LinearSVC())
])

full_pipeline.fit(x_train, y_train)

y_pred_train = full_pipeline.predict(x_train)
print(f'training score: {f1_score(y_train, y_pred_train)}')

y_pred = full_pipeline.predict(x_val)
print(f'validation score: {f1_score(y_val, y_pred)}')


training score: 0.5949506037321625
validation score: 0.5502183406113537


- RandomForestClassifier:  
training score: 1.0  
validation score: 0.5253456221198156 --> holy overfitting  

- KNeighborsClassifier  
training score: 0.6857825567502986  
validation score: 0.5454545454545454  

- LinearSVC  
training score: 0.5949506037321625  
validation score: 0.5502183406113537  

In [72]:
num_pipe = Pipeline(steps=[
    (
        'num_imputer',
        SimpleImputer(strategy='median', add_indicator=True)
    )
    ,
    (
        'scaler',
        StandardScaler()
    )
])

ord_pipe = Pipeline(steps=[
    (
        'ord_imputer',
        SimpleImputer(strategy='constant', fill_value='unknown')
    )
    ,
    (
        'OE',
        OrdinalEncoder(categories=categories, handle_unknown='use_encoded_value', unknown_value=-1, encoded_missing_value=-2)
    )
])

cat_pipe = Pipeline(steps=[
    (
        'cat_imputer',
        SimpleImputer(strategy='constant', fill_value='unknown', add_indicator=True)
    )
    ,
    (
        'OHE',
        OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    )
])

preprocesing_pipeline = ColumnTransformer(transformers=[
    ('num_prep', num_pipe, num_col_name),
    ('ord_prep', ord_pipe, ord_col_name),
    ('cat_prep', cat_pipe, cat_col_name)
], remainder='passthrough', n_jobs=-1)

full_pipeline = Pipeline(steps=[
    ('preprocessing', preprocesing_pipeline),
    ('classifier', RandomForestClassifier(n_jobs=-1, random_state=42))
])

param_grid = {
    'classifier__n_estimators' : [100, 150, 200],
    'classifier__max_depth' : [None, 10, 20],
    'classifier__min_samples_split' : [2,5,10],
    'classifier__min_samples_leaf' : [3,5]
}

grid = GridSearchCV(estimator=full_pipeline, param_grid=param_grid, scoring='f1', cv=3, n_jobs=-1, verbose=1)
grid.fit(x_train, y_train)

print(grid.best_params_)
best_model = grid.best_estimator_

y_pred_train = best_model.predict(x_train)
print(f'training score: {f1_score(y_train, y_pred_train)}')

y_pred = best_model.predict(x_val)
print(f'validation score: {f1_score(y_val, y_pred)}')

Fitting 3 folds for each of 54 candidates, totalling 162 fits
{'classifier__max_depth': None, 'classifier__min_samples_leaf': 5, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 150}
training score: 0.8319088319088319
validation score: 0.8532110091743119


In [ ]:
y_pred_df = pd.DataFrame({'ID': len(range(y_pred)), 'prediction': y_pred})
y_pred_df.to_csv('predictions.csv', index=False)

---

# MAIN

In [74]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score

def get_data():
    train_path = '/Users/filippomontecchi/Desktop/Data Science & Machine Learning Lab/Lab/Code/CORE_CODING/classification/toy_dataset/classification_dataset.csv'
    train_df = pd.read_csv(train_path)
    
    x_train = train_df.loc[:, 'num_1':'cat_2']
    y_train = train_df['target']
    
    return x_train, y_train


def preprocessing_training(x_train, y_train):
    num_col = x_train.loc[:, 'num_1':'num_2']
    ord_col = x_train.loc[:, 'ord_1':'ord_2']
    cat_col = x_train.loc[:, 'cat_1':'cat_2']

    num_col_name = num_col.columns.tolist()
    ord_col_name = ord_col.columns.tolist()
    cat_col_name = cat_col.columns.tolist()

    categories = [
        ['High', 'Medium', 'Low'],  # ord_1 categories
        ['Poor', 'Average', 'Good'] # ord_2 categories
    ]

    # drop columns and rows here
    # --> droop in x_train and y_train BUT DO NOT DO IT FOR VALIDATION OR TEST
    # --> for test and validation you drop columns ONLY otherwise the predicted labels get disaligned with the professor's ones


    num_pipe = Pipeline(steps=[
        (
            'num_imputer',
            SimpleImputer(strategy='median', add_indicator=True)
        )
        ,
        (
            'scaler',
            StandardScaler()
        )
    ])

    ord_pipe = Pipeline(steps=[
        (
            'ord_imputer',
            SimpleImputer(strategy='constant', fill_value='unknown')
        )
        ,
        (
            'OE',
            OrdinalEncoder(categories=categories, handle_unknown='use_encoded_value', unknown_value=-1, encoded_missing_value=-2)
        )
    ])

    cat_pipe = Pipeline(steps=[
        (
            'cat_imputer',
            SimpleImputer(strategy='constant', fill_value='unknown', add_indicator=True)
        )
        ,
        (
            'OHE',
            OneHotEncoder(sparse_output=False, handle_unknown='ignore')
        )
    ])

    preprocesing_pipeline = ColumnTransformer(transformers=[
        ('num_prep', num_pipe, num_col_name),
        ('ord_prep', ord_pipe, ord_col_name),
        ('cat_prep', cat_pipe, cat_col_name)
    ], remainder='passthrough', n_jobs=-1)

    full_pipeline = Pipeline(steps=[
        ('preprocessing', preprocesing_pipeline),
        ('classifier', RandomForestClassifier(max_depth=None, min_samples_split=2, min_samples_leaf=5, n_estimators=150, n_jobs=-1, random_state=42))
    ])

    full_pipeline.fit(x_train, y_train)

    y_pred_train = full_pipeline.predict(x_train)
    print(f'training score: {f1_score(y_train, y_pred_train)}')

    return full_pipeline

# def make_predictions(x_test, full_pipeline):
#     # drop eventual columns you previously dropped in x_train and y_train
#     # BUT NOT ROWS!!
#     y_pred = full_pipeline.predict(x_test)
#     y_pred_df = pd.DataFrame({'ID': range(len(y_pred)), 'predictions': y_pred})
#     y_pred_df.to_csv('predictions.csv', index=False)


if __name__ == '__main__':
    x_train, y_train = get_data()
    full_pipeline = preprocessing_training(x_train, y_train)
    





training score: 0.8319088319088319
